# 1. Python tools

In this lesson, you will build a Python function that looks up a support ticket and returns its status as structured data. Along the way, you will practise collections, functions, type hints, classes, imports, JSON, and error handling.

Open this notebook with a Python 3.11 or newer kernel and run the cells in order with **Shift + Enter**. Predict the output before running each example, then compare it with the expected results. The file examples write small demo files to the kernel's working folder. Use `Path.cwd()` to find that folder.

[Environment setup](../01-introduction/README.md) · [Python basics](../02-python-basics/python_basics.ipynb) · [Next lesson: Data, charts and LLMs](02_data_and_llms.ipynb)

## Setup, variables and types

A support ticket records a customer's request for help. Run the next cell to check which version of Python the notebook is using. If you need help setting up the kernel, see the [environment setup notes](../01-introduction/README.md).

In [1]:
import sys

print(sys.version.split()[0])
print("Python is ready")

3.11.4
Python is ready


The first line imports Python's built-in `sys` module. The next two lines show the Python version and confirm the cell ran. If the version is older than 3.11, select a newer kernel.

In [2]:
ticket_id = 101
message = "I cannot log in"
wait_minutes = 42.5
is_open = True
assigned_to = None

print(type(ticket_id).__name__, type(message).__name__)
print(type(wait_minutes).__name__, type(is_open).__name__)
print(assigned_to)

int str
float bool
None


A variable is a name we give a value so we can use that value again. A ticket ID is a whole number. The message is text. Wait time can have a decimal. A boolean is true or false. Here, `None` means nobody has been assigned yet.

**Expected** — `int str`, then `float bool`, then `None`. `type(value).__name__` gives the type's name.

**Check your understanding** — If a form gives us `"101"`, does it equal the integer `101`?  
**Answer** — The two values have different types. Convert a valid numeric string with `int("101")`. Show how text and integers behave differently with `+`.

In [3]:
print("101" + "1")
print(int("101") + 1)

1011
102


The first result is `1011` because Python joins two strings. The second is `102` because Python adds two integers.

## Decisions and loops

A condition is a question with a true or false answer. The order matters. Check whether the ticket is closed before checking how long it has been waiting.

`=` assigns a value. `==` compares two values. `elif` checks another condition if the earlier one was false. Indentation tells Python which lines belong to each branch.

In [4]:
status = "open"
wait_minutes = 42.5

if status == "closed":
    print("No action needed")
elif wait_minutes >= 30:
    print("Send to the priority queue")
else:
    print("Keep in the standard queue")

Send to the priority queue


**Expected** — `Send to the priority queue`.

**Check your understanding** — What happens at exactly 30 minutes? What if a closed ticket has waited 60 minutes?  
**Answer** — Thirty qualifies because we used `>=`. A closed ticket uses the first branch and skips the others.

**Try it** — Change the rule to 45 minutes. Test 42.5 and 45 with an open ticket. The expected routes are standard and priority.

A `for` loop takes one item at a time from a collection. We can skip a blank message before sending anything to a tool. That avoids doing work on empty input.

In [5]:
messages = ["  Cannot log in  ", "  ", "Need an invoice"]
clean_messages = []

for message in messages:
    cleaned = message.strip()
    if cleaned == "":
        continue
    clean_messages.append(cleaned)

print(clean_messages)

['Cannot log in', 'Need an invoice']


`.strip()` removes whitespace at the ends. `continue` skips the rest of the current round. `.append()` adds one item to the list. Expect `['Cannot log in', 'Need an invoice']`.

Use a `while` loop when the work continues while a condition stays true. Imagine a later agent with a limit on how many actions it can try. The limit gives the loop a clear stopping point.

In [6]:
steps_left = 3

while steps_left > 0:
    print("A step is available")
    steps_left -= 1

print("Step limit reached")

A step is available
A step is available
A step is available
Step limit reached


The loop runs three times. Each round subtracts one from the counter. A `break` would leave the loop immediately, for example once a search finds a match.

**Check your understanding** — What happens if you remove the subtraction? The counter stays the same and the condition stays true.

## Collections, functions and type hints

A collection lets us keep several related values together. Before choosing one, ask how you need to use the data. Do you care about position? Do you want a field by name? Do you need to keep only unique values? In our support desk, a list holds several tickets in order. A dictionary describes one ticket using fields such as ID and status.

| Collection | When to choose it | Create an empty one |
| --- | --- | --- |
| List | Keep items in order, including duplicates, and change the list as work arrives. | `[]` |
| Dictionary | Look up a value using a named key. Each key has one current value. | `{}` |
| Tuple | Keep a fixed group of values in order, such as opening and closing hours. | `()` |
| Set | Keep distinct values or quickly check whether a value has already been seen. | `set()` |

In [7]:
tickets = [
    {"id": 101, "category": "login", "status": "open"},
    {"id": 102, "category": "billing", "status": "closed"},
    {"id": 103, "category": "login", "status": "open"},
]

office_hours = (9, 18)
categories = set()

for ticket in tickets:
    categories.add(ticket["category"])

print(tickets[0]["status"])
print(office_hours[1])
print(sorted(categories))

open
18
['billing', 'login']


Read the example from the inside out. `tickets[0]` gets the first ticket, which is a dictionary. The next brackets get the value under the key `"status"`. Office hours has two positions, so index 1 gets the second value. We use `.add()` to collect each distinct category.

**Expected** — `open`, `18`, and `['billing', 'login']`. The loop sees `"login"` twice, but the set keeps it once. The output looks alphabetical because `sorted(categories)` builds a new sorted **list**. The set has no guaranteed order, so use `sorted()` whenever you want a predictable display. For these lowercase strings, the default sort puts `billing` before `login`.

**Check your understanding** — Why use a dictionary for one ticket?  
**Answer** — The keys tell us what each value means. The outer list lets us keep many of those records in order.

### List · Create, read, update and add

Use a list when the order matters. Python starts counting at zero. Index zero is the first item, and minus one gets the last. A slice gives us part of the list, and its stop position is left out. We can change an existing position or add more items.

**Teach first**

- `append(item)` adds one item at the end.
- `extend(items)` adds each item from another collection.
- `insert(index, item)` puts one item before the given index and shifts the later items.

In [8]:
pending = [101, 103]

print(pending[0], pending[-1])
print(pending[:1])
print(len(pending), 103 in pending)

pending[1] = 104
pending.append(105)
pending.extend([106, 107])
pending.insert(0, 100)
print(pending)

101 103
[101]
2 True
[100, 101, 104, 105, 106, 107]


**Expected** — The first line shows `101 103`. The slice is `[101]`, and the length and membership check show `2 True`. The final list is `[100, 101, 104, 105, 106, 107]`.

**Check your understanding** — What would `pending.append([106, 107])` add?  
**Answer** — One item, which happens to be a list. `extend([106, 107])` adds two separate IDs. An agent might use `append()` for one completed action and `extend()` to add a batch of search results.

### List · Remove by value or position

Say which thing you know before choosing the operation. If you know the ticket ID, remove by value. If you know the position and need the item back, use pop. If you only want to delete a position, use del.

- `remove(value)` removes the first match. It raises `ValueError` if the value is absent.
- `pop()` removes and returns the last item.
- `pop(0)` removes and returns the first item.
- `del items[index]` deletes an item by position.

In [9]:
pending = [101, 102, 103, 102, 104]

pending.remove(102)
print(pending)

next_ticket = pending.pop(0)
print(next_ticket, pending)

del pending[1]
print(pending)

[101, 103, 102, 104]
101 [103, 102, 104]
[103, 104]


**Expected** — `[101, 103, 102, 104]`, then `101 [103, 102, 104]`, then `[103, 104]`.

**Optional · Search, count, copy and sort**

- `in` checks whether a value exists in the list.
- `.index(value)` gives the first matching position. It raises `ValueError` if the value is absent.
- `.count(value)` counts how many times the value appears.
- `sorted(items)` returns a new sorted list.
- `.sort()` sorts the list in place and returns `None`.
- `.reverse()` reverses the list in place and returns `None`.
- `.copy()` makes a separate outer list.
- `.clear()` removes all items from the list.

In [10]:
pending = [103, 101, 103, 102]
print(pending.index(103), pending.count(103))
print(sorted(pending), pending)

another_queue = pending.copy()
another_queue.sort()
another_queue.reverse()
print(another_queue, pending)

0 2
[101, 102, 103, 103] [103, 101, 103, 102]
[103, 103, 102, 101] [103, 101, 103, 102]


**Expected** — `0 2`, then a sorted list next to the original, then `[103, 103, 102, 101] [103, 101, 103, 102]`.

**Check your understanding** — What happens if we write `pending = pending.sort()`?  
**Answer** — `.sort()` sorts the original list and returns `None`, so the name `pending` now points to `None`. Write `pending.sort()` on its own, or use `pending = sorted(pending)` to assign a new sorted list.

### Dictionary · Read and change named fields

Use a dictionary when each piece of data has a name. A ticket can have keys such as ID, category and status. To change a field, assign a value to its key. The same syntax creates a field when that key is new.

- `record[key]` reads a value by key. A missing key raises `KeyError`.
- `record.get(key, default)` returns the default when the key is absent.
- `record.pop(key)` removes the key and returns its value.
- `record.pop(key, default)` returns the default when the key is absent.
- `del record[key]` deletes the entry by key.

In [11]:
record = {"id": 101, "status": "open"}

print(record["id"], record.get("priority", "unknown"))
record["category"] = "login"
record["status"] = "closed"
old_category = record.pop("category")
print(old_category, record)

del record["status"]
print("status" in record, record)

101 unknown
login {'id': 101, 'status': 'closed'}
False {'id': 101}


**Expected** — `101 unknown`, then `login {'id': 101, 'status': 'closed'}`, then `False {'id': 101}`. The `in` check looks for a **key**.

**Optional · Update and loop over fields**

- `.update()` adds new keys and overwrites values for matching keys.
- `.keys()` gives the keys.
- `.values()` gives the values.
- `.items()` gives key-value pairs.
- `.clear()` removes all entries.

In [12]:
record = {"id": 101, "status": "open"}
record.update({"status": "closed", "priority": "high"})

print(list(record.keys()))
print(list(record.values()))
for key, value in record.items():
    print(key, value)

['id', 'status', 'priority']
[101, 'closed', 'high']
id 101
status closed
priority high


**Check your understanding** — Which would you use for a tool request with a required ID and an optional priority?  
**Answer** — Read and validate the ID as a required field. Use `.get("priority", "normal")` for the optional field. The loop above unpacks each key-value pair into two names.

### Tuple · Keep a fixed group of values

A tuple is useful when a group of values belongs together and its positions should stay fixed. Office hours always have an opening time and a closing time. We can read those values by position or give each one a name by unpacking the tuple.

- Indexing and slicing read items or a portion of the tuple.
- `len()` gives the number of items.
- `in` checks whether an item is present.
- `.count(value)` counts occurrences of a value.
- `.index(value)` gives the first matching position. It raises `ValueError` if the value is absent.
- Tuple slots cannot be changed, so there are no `.append()` or `.remove()` methods. Build a new tuple when you need different items.

In [13]:
office_hours = (9, 18)
opening, closing = office_hours

print(opening, closing)
print(office_hours.count(9), office_hours.index(18))

saturday_hours = (9, 13)
print(saturday_hours, office_hours)

9 18
1 1
(9, 13) (9, 18)


**Expected** — `9 18`, then `1 1`, then `(9, 13) (9, 18)`. The second tuple represents different hours. The first tuple's slots stay as they were.

**Optional · One value and a new tuple**

- `(9,)` is a one-item tuple. The comma makes it a tuple.
- `(9)` is an integer in parentheses.

In [14]:
opening_hours = (9,)
office_hours = opening_hours + (18,)

print(opening_hours, office_hours)
print(office_hours[:1])

(9,) (9, 18)
(9,)


**Expected** — `(9,) (9, 18)`, then `(9,)`. Joining tuples and slicing tuples each give a new tuple. If the group will grow and shrink often, use a list.

### Set · Keep unique items and check membership

Use a set when you care whether something has been seen. Imagine an agent collects ticket IDs from two tools. It can keep a set of IDs it has already handled. Adding the same ID twice still leaves one entry.

- `set()` creates an empty set. `{"login", "billing"}` creates a set with values.
- `.add(value)` adds one item.
- `.update(items)` adds each item from a collection.
- `.discard(value)` removes the value if it is present. It does nothing if the value is absent.
- `.remove(value)` removes the value and raises `KeyError` if it is absent.
- A set has no positions to insert at or index into.

In [15]:
categories = {"login", "billing"}
categories.add("login")
categories.update(["shipping", "billing"])
print(sorted(categories))

categories.discard("returns")
categories.remove("shipping")
print("login" in categories)
print(sorted(categories))
print(type(sorted(categories)).__name__)

['billing', 'login', 'shipping']
True
['billing', 'login']
list


**Expected** — `['billing', 'login', 'shipping']`, `True`, `['billing', 'login']`, then `list`. The set handles uniqueness. `sorted()` handles ordering.

**Optional · Combine or compare two sets**

- `.union()` gives values in either set.
- `.intersection()` gives values shared by both sets.
- `first.difference(second)` gives values in the first set that are absent from the second.
- `.clear()` removes all items from the set.

In [16]:
found_ids = {101, 102, 103}
handled_ids = {102, 104}

print(sorted(found_ids.union(handled_ids)))
print(sorted(found_ids.intersection(handled_ids)))
print(sorted(found_ids.difference(handled_ids)))

[101, 102, 103, 104]
[102]
[101, 103]


**Expected** — `[101, 102, 103, 104]`, `[102]`, and `[101, 103]`.

**Check your understanding** — A tool returns `[103, 101, 103]`. What would you keep if you need the original order of results?  
**Answer** — Keep the list. If you also need to avoid processing an ID twice, keep a set of seen IDs and walk through the list in order.

### Functions · Give a piece of work a name

A function is a named piece of work. We write the steps once and call them whenever we need the result. This makes it easier to test one job at a time. For example, two screens might send us categories written in different ways. Both can use the same function to clean the text.

**How to read the code**

- `def` defines the function.
- The indented body runs when we call the function with parentheses.
- Names such as `cleaned` hold intermediate results inside the function.
- `return` gives a result back to the caller and ends that call.

**Agent connection** — A tool can be a Python function that the application makes available for a specific job. Its name and description tell a caller what it does, its parameters describe what it needs, and its return value provides the answer.

In [17]:
def normalize_category(category: str) -> str:
    cleaned = category.strip()
    return cleaned.lower()


category = normalize_category("  BILLING  ")
print(category)

billing


A type hint tells the next person what kind of value is expected. Here the function expects text and returns text. Think of the hints as part of the function's instructions. Python still needs a check if the program must reject the wrong type at runtime.

**Walk through it**

- `category` in the definition is the parameter.
- `"  BILLING  "` is the argument passed in the call.
- `.strip()` removes spaces at the ends.
- `.lower()` turns letters lowercase.
- `: str` describes the input type, and `-> str` describes the return type.

Expect `billing`.

**Check your understanding** — Why use `return` inside the function?  
**Answer** — The next line of our program can store and use the result. If the function only prints the value, the caller receives `None`.

**Optional · Defaults and keyword arguments.**

Some inputs have a sensible default. Our example considers a wait of 30 minutes long enough to review. We can call the function with just the wait or pass a different limit. Naming the argument makes it clear what the 45 means.

- A **default value** is used when that argument is left out.
- A **keyword argument** supplies an argument by parameter name.

In [18]:
def needs_review(wait_minutes: int, threshold_minutes: int = 30) -> bool:
    return wait_minutes >= threshold_minutes


print(needs_review(40))
print(needs_review(40, threshold_minutes=45))

True
False


**Expected** — `True`, then `False`. The `>=` comparison includes the threshold itself. Try predicting what `needs_review(30)` returns.

**Optional · Lambda functions**

The support lead wants to see the longest wait first. Every ticket has a wait time. We can give `sorted()` a small function that picks the wait time from each ticket. A lambda lets us write that function right where we use it.

- `lambda ticket: ticket["wait_minutes"]` takes one ticket and gives back its wait.
- A lambda contains one expression. Python returns its value automatically.
- `key=` gives `sorted()` the function to use for each ticket.
- `reverse=True` puts larger waits first.

In [19]:
waiting_tickets = [
    {"id": 101, "wait_minutes": 15},
    {"id": 102, "wait_minutes": 45},
    {"id": 103, "wait_minutes": 30},
]

longest_first = sorted(
    waiting_tickets,
    key=lambda ticket: ticket["wait_minutes"],
    reverse=True,
)

for ticket in longest_first:
    print(ticket["id"], ticket["wait_minutes"])

102 45
103 30
101 15


**Expected** — `102 45`, then `103 30`, then `101 15`.

**Check your understanding** — What happens if we remove `reverse=True`?  
**Answer** — The 15-minute ticket comes first, then 30 and 45.

**Common question · Can I use `def` here?**  
**Answer** — Yes. Write the function below and pass `key=get_wait`. Use a named function when you plan to reuse the logic or need several steps.

```python
def get_wait(ticket: dict[str, int]) -> int:
    return ticket["wait_minutes"]
```

**Agent connection** — An application can sort the tickets returned by a tool before it shows them to a support lead.

In [20]:
def get_ticket_status(
    records: list[dict[str, int | str]], ticket_id: int
) -> str:
    '''Return the status for a known ticket ID.'''
    if type(ticket_id) is not int or ticket_id <= 0:
        raise ValueError("Ticket ID must be a positive integer.")

    for ticket in records:
        if ticket["id"] == ticket_id:
            return str(ticket["status"])

    return "not_found"


print(get_ticket_status(tickets, 102))
print(get_ticket_status(tickets, 999))

closed
not_found


In [21]:
from typing import get_type_hints

print(get_type_hints(get_ticket_status))

{'records': list[dict[str, int | str]], 'ticket_id': <class 'int'>, 'return': <class 'str'>}


In [22]:
import inspect

print(inspect.signature(get_ticket_status))

(records: list[dict[str, int | str]], ticket_id: int) -> str


Read the lookup function in three steps. First it checks the input. Next it searches the records one by one. If it finds a match, it returns that ticket's status immediately. If the loop finishes without a match, it returns the text we chose for that case.

`list[dict[str, int | str]]` describes a list of dictionaries with string keys and integer or string values. The `|` means either type. The first string inside the function is a **docstring**, a short description of the job. `raise ValueError(...)` signals that the caller gave an invalid ID. A positive ID that happens to be absent is a separate case. Expect `closed` for 102 and `not_found` for 999.

**Try it** — Add a record with a new ID. Predict the results for that ID and an absent ID before running the calls.

**Check your understanding** — Where does `return "not_found"` belong?  
**Answer** — After the loop. If it sits inside the loop, the first mismatch could end the search before we reach the correct ticket.

## OOP with a ticket queue

OOP stands for object-oriented programming. It is a way of organizing a program around objects. An object can keep information and provide actions that use that information. Think of two support teams. Each team has its own queue, but both teams know how to add a ticket and count the tickets waiting.

Why make a class? We could use two lists and a few functions. As the queue gains a name, rules, and more actions, a class gives us one place to describe how every queue works. Each queue still keeps its own data. For one small job like cleaning a string, a function is often enough.

| Term | Meaning in this example |
| --- | --- |
| Class | `TicketQueue` describes what a queue stores and what it can do. |
| Object or instance | `login_queue` and `billing_queue` are two objects made from that class. |
| Attribute | `name` and `ticket_ids` are values stored on each object. |
| Method | `add()` and `count()` are functions we call on a queue. |
| State | The current values held by one queue, such as its list of ticket IDs. |

**How it works** — `TicketQueue("login")` creates an object. Python calls `__init__` to give it a name and a fresh list. When we call a method on an object, Python passes that object in as the first argument, which we name `self`.

In [23]:
class TicketQueue:
    def __init__(self, name: str):
        self.name = name
        self.ticket_ids: list[int] = []

    def add(self, ticket_id: int) -> None:
        self.ticket_ids.append(ticket_id)

    def count(self) -> int:
        return len(self.ticket_ids)


login_queue = TicketQueue("login")
billing_queue = TicketQueue("billing")

login_queue.add(101)
login_queue.add(103)

print(login_queue.name, login_queue.count())
print(billing_queue.name, billing_queue.count())

login 2
billing 0


**Expected** — `login 2` and `billing 0`.

Follow one call with me. In `login_queue.add(101)`, `self` refers to `login_queue` and `ticket_id` is 101. The method appends 101 to that queue's own list. When I call `billing_queue.count()`, `self` now refers to the billing queue. Its list is still empty.

The list is created inside `__init__`, so each object gets a new one. `add()` changes the object and returns `None`. `count()` reads the object's current state and returns an integer.

**Try it** — Add ticket 102 to the billing queue. Predict the counts before running. Expect 2 and 1.

**Check your understanding** — Where could an agent application use a class?  
**Answer** — A ticket-system client could store a server address and expose methods to look up tickets. A workflow could also keep its running history on an object. Each instance can store data for a different team or run.

**Optional · Give the class one clear operation.**

Suppose we want to remember which tickets we have seen. We can give callers two clear methods, one to mark a ticket and another to check it. Keeping the stored data and the operations on it together is an example of encapsulation.

This separate example uses a set for the IDs, so a repeated ID has no effect. A real queue that must process tickets in arrival order would keep a list for the order.

In [24]:
class SeenTickets:
    def __init__(self):
        self.ids: set[int] = set()

    def mark_seen(self, ticket_id: int) -> None:
        self.ids.add(ticket_id)

    def has_seen(self, ticket_id: int) -> bool:
        return ticket_id in self.ids


seen = SeenTickets()
print(seen.has_seen(101))
seen.mark_seen(101)
seen.mark_seen(101)
print(seen.has_seen(101), len(seen.ids))

False
True 1


**Expected** — `False`, then `True 1`. Connect this to a later agent workflow that needs to remember which tickets it has already inspected.

## Modules and imports

A module is a unit of Python code that we can import. A simple module is one `.py` file. If the same ticket function is useful in three notebooks, we can keep it in one module and import it in all three places. When we fix the function, each caller can use that same fix.

When we import a module, Python makes its code available. We can import the whole module and write its name before a member, as in `pathlib.Path`. Or we can import the name we need with `from pathlib import Path` and then write `Path` directly.

**Agent connection** — A later application can keep tools such as ticket lookup in one module and keep the code that chooses and runs those tools in another.

In [25]:
from pathlib import Path

path = Path("support_report.json")
print(path.name)
print(path.suffix)

support_report.json
.json


`pathlib` is a module included with Python. `Path` is the class we imported from it. The example prints the filename and its suffix. It has only created a path object in memory so far.

**Try in VS Code** — In a scratch folder, make these two files side by side. Run `python demo.py` from that folder using your Python environment. Then return to the notebook.

**In `ticket_tools.py`**

```python
def normalize_category(category: str) -> str:
    return category.strip().lower()
```

**In `demo.py`**

```python
from ticket_tools import normalize_category

print(normalize_category("  LOGIN  "))
```

**Expected** — `login`. Python imports by module name, so omit `.py`.

**Check your understanding** — Why do this for a tool?  
**Answer** — The live demo, tests, and a later agent can all use the same function from one file.

## Files, JSON and exceptions

### Files · Keep a result after the program ends

A variable lives in the Python session. If we want to open the result later, we can save it to a file. A path says where that file lives. Opening a file gives us something Python can read from or write to.

- `"w"` opens a file for writing and replaces its contents if it already exists.
- `"r"` opens a file for reading.
- `"a"` opens a file for appending text at the end.
- `with` closes the file when the block ends, including when an error occurs.
- `encoding="utf-8"` tells Python how to read and write the text.

The example writes `ticket_result.json` in the notebook kernel’s current working folder. Run `print(Path.cwd())` to see that folder. Running the writing cell again replaces the file’s contents.

### JSON · Share structured data

A dictionary is a Python object. JSON is a text format other programs can understand. We can turn a dictionary into JSON, save or send the text, and read it back as Python values. An API may send tool arguments as JSON. A tool can also write its result as JSON so another program can read the fields.

In the next cell, `json.dump()` writes our dictionary to an open file. `json.load()` reads the JSON in that file and gives us back a Python dictionary. The `indent=2` setting makes the saved text easier to read.

In [26]:
import json
from pathlib import Path

report_path = Path("ticket_result.json")
result = {"ticket_id": 101, "status": get_ticket_status(tickets, 101)}

with report_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, indent=2)

with report_path.open("r", encoding="utf-8") as file:
    saved_result = json.load(file)

print(saved_result["status"])
print(report_path.read_text(encoding="utf-8"))

open
{
  "ticket_id": 101,
  "status": "open"
}


**Expected** — `open` and the JSON for ticket 101. The first print reads a value from the dictionary that came back from the file. The second print shows the file's text.

| Operation | File | String |
| --- | --- | --- |
| Convert Python values to JSON | `json.dump(value, file)` | `json.dumps(value)` |
| Read JSON into Python values | `json.load(file)` | `json.loads(text)` |

**Optional · Show the difference in memory**

- The final `s` in `dumps` and `loads` is a reminder that these versions work with a string.

In [27]:
sample_result = {"ticket_id": 101, "found": True, "owner": None}
result_text = json.dumps(sample_result)

print(type(result_text).__name__, result_text)
restored_result = json.loads(result_text)
print(type(restored_result).__name__, restored_result["found"])

str {"ticket_id": 101, "found": true, "owner": null}
dict True


**Expected** — The JSON string contains `true` and `null`. After reading it back, Python prints `dict True`.

**Check your understanding** — Why would we change a set to a list before including it in a JSON tool result?  
**Answer** — JSON has arrays and the standard encoder writes Python lists as arrays. It has no set type. Use `sorted(categories)` if you also want predictable ordering.

### Exceptions · Handle a failure you expect

An exception is how Python signals a problem while code is running. A file can be missing, a number can arrive as letters, or a reply can contain broken JSON. We use try and except when we know how to respond to a particular failure.

Put the work that may fail in the try block. If Python raises a matching exception, it jumps to that except block. If the work succeeds, Python skips the except blocks. Give each known failure a clear message so the next person knows what happened.

The next cell tries to open `missing.json` in the current working folder. If a file with that name already exists, change the name in the cell to one that does not exist. The two exception names describe different kinds of failure.

- `FileNotFoundError` is raised when the path does not exist.
- `json.JSONDecodeError` is raised when the file is readable but its contents are invalid JSON.

In [28]:
try:
    with report_path.with_name("missing.json").open("r", encoding="utf-8") as file:
        report = json.load(file)
    print(report)
except FileNotFoundError:
    print("No report file was found.")
except json.JSONDecodeError:
    print("The report file does not contain valid JSON.")

No report file was found.


**Expected** — `No report file was found.` Python jumps to the matching handler before it gets to `print(report)`.

**Optional · Make the second handler run**

- This example writes a file named `broken_report.json` in the current working folder with text that JSON cannot parse. It replaces the contents if that file already exists.

In [29]:
bad_report_path = report_path.with_name("broken_report.json")
bad_report_path.write_text("oops", encoding="utf-8")

try:
    with bad_report_path.open("r", encoding="utf-8") as file:
        bad_report = json.load(file)
except json.JSONDecodeError:
    print("The report file does not contain valid JSON.")

The report file does not contain valid JSON.


**Expected** — `The report file does not contain valid JSON.`

**Check your understanding** — What should a tool report when it cannot read its data?  
**Answer** — Say that the data could not be read. An empty list means we read the data and found no records. The application can use the reason for the failure to decide its next step.

## Practise a tool call

A tool receives data from another part of the application. That data might be a number or a string. We decide what the tool accepts and where conversion happens. In this example the tool expects a positive integer. Code that reads text from a form can convert it before calling the tool.

**Practice** — A tool receives the ticket ID `"102"` as a string. First call `get_ticket_status()` with that value inside `try` and catch `ValueError`. Then convert the string with `int()` and call the tool again. Predict both outcomes before running them.

In [30]:
try:
    print(get_ticket_status(tickets, "102"))
except ValueError as error:
    print(error)

print(get_ticket_status(tickets, int("102")))

Ticket ID must be a positive integer.
closed


Expect the validation message, then `closed`. Think about where to handle a string like `"ABC"`. `int("ABC")` itself raises a `ValueError`, so include that conversion in the `try` when reading form input.

Imagine a user asks, “Is ticket 101 still open?” An agent could request the ticket-status tool with ID 101. Python checks the request and looks up the status. The model can then use that result to answer in a sentence. Today we'll supply the request ourselves.

In [31]:
request_text = '{"tool": "get_ticket_status", "ticket_id": 101}'
tool_request = json.loads(request_text)

if tool_request["tool"] == "get_ticket_status":
    status = get_ticket_status(tickets, tool_request["ticket_id"])
    tool_result = {"ticket_id": tool_request["ticket_id"], "status": status}
    print(json.dumps(tool_result))
else:
    print("Unknown tool")

{"ticket_id": 101, "status": "open"}


**Expected** — `{"ticket_id": 101, "status": "open"}`. The `if` picks the supported tool name. The function looks up the ticket. The dictionary gives the caller named fields it can use.

Follow the data. We start with request text. We read the JSON into a dictionary, check which tool the request names, and pass the ID to our lookup function. Then we turn the result into JSON. A later agent could read those fields and say, “Ticket 101 is still open.”

**Check your understanding** — Which line runs the tool? Where would a model enter this flow?  
**Answer** — Python runs it on the `get_ticket_status(...)` line. In a later agent, the model can choose the tool and propose arguments, then use the returned data. This demo uses a fixed request.

**Optional practice**

- Rerun the last cell with ID 999 to show `not_found`.
- Change the name to `"close_ticket"` to show `Unknown tool`.
- Explain why the two requests lead to different outcomes.

**Exit check** — Identify the input, the validation, and the output of the tool. In the [next lesson](02_data_and_llms.ipynb), you will use a table of tickets to write a support update.

[July batch Python for AI roadmap](https://skarma91.github.io/logicmojo-ai-july-2026/#module-1)

## End-of-class recap

### `json.dump()` vs `json.dumps()`

Both turn a Python value into JSON. Ask where you want that JSON to go. If another program needs the saved file, use dump with a file you opened for writing. If you want JSON text in a variable, use dumps.

- `json.dump(value, file)` writes JSON to an open file. It returns `None`.
- `json.dumps(value)` returns JSON text as a Python string.
- `json.load(file)` reads JSON from an open file.
- `json.loads(text)` parses JSON text already in a string.

In [32]:
import json

ticket = {"id": 101, "status": "open"}
json_text = json.dumps(ticket)
print(type(json_text).__name__, json_text)

with open("recap_ticket.json", "w", encoding="utf-8") as file:
    json.dump(ticket, file, indent=2)

with open("recap_ticket.json", "r", encoding="utf-8") as file:
    saved_ticket = json.load(file)

print(saved_ticket["status"])

str {"id": 101, "status": "open"}
open


**Expected** — `str {"id": 101, "status": "open"}`, followed by `open`.

**Check your understanding** — Which function gives us text we can put in a variable and pass to another part of the application?  
**Answer** — `json.dumps()`. Its return value is a string.

### Append to a file or update its contents

Appending means adding something at the end. Updating an existing value takes three steps. Read the existing data, change the value in Python, and write the updated data back.

| Mode | What it does |
| --- | --- |
| `"r"` | Reads an existing file. A missing file raises `FileNotFoundError`. |
| `"w"` | Creates a file or clears an existing file as soon as it is opened. |
| `"a"` | Creates a file if needed and adds each write to the end. |
| `"r+"` | Opens an existing file for reading and writing without clearing it first. Writing can overwrite existing content at the current position. |

The recap uses files in the notebook kernel’s current working folder. The setup cells open them in `"w"` each time.

**Append a log entry** — Run these blocks together. The first creates the sample log, the second adds one line, and the third reads both lines.

In [33]:
with open("recap_ticket_log.txt", "w", encoding="utf-8") as file:
    file.write("Ticket 101 opened\n")

with open("recap_ticket_log.txt", "a", encoding="utf-8") as file:
    file.write("Ticket 101 assigned\n")

with open("recap_ticket_log.txt", "r", encoding="utf-8") as file:
    print(file.read(), end="")

Ticket 101 opened
Ticket 101 assigned


**Expected** — `Ticket 101 opened` followed by `Ticket 101 assigned` on the next line. Rerun only the append block and the read block to show the new line being added a second time.

**Update an existing JSON field** — Run this after section 2. The file holds one JSON object, so load that object, update its status, and write the full object back. Appending a second JSON object would leave two objects in a file that `json.load()` expects to contain one.

In [34]:
with open("recap_ticket.json", "r", encoding="utf-8") as file:
    ticket = json.load(file)

ticket["status"] = "closed"

with open("recap_ticket.json", "w", encoding="utf-8") as file:
    json.dump(ticket, file, indent=2)

with open("recap_ticket.json", "r", encoding="utf-8") as file:
    print(json.load(file))

{'id': 101, 'status': 'closed'}


**Expected** — `{'id': 101, 'status': 'closed'}`.

**Check your understanding** — What happens if we open the JSON file in `"w"` before reading it?  
**Answer** — Opening in `"w"` clears the old contents immediately. Read first, then open it for writing.

### Public, protected and private names in a Python class

A class can have public features that other code is meant to use. It can also keep helper methods and attributes for its own work. Python uses names to communicate how each member is intended to be used.

| Naming style | Attribute example | Method example | Meaning |
| --- | --- | --- | --- |
| Public | `name` | `add_ticket()` | Other code is meant to use it. |
| One leading underscore | `_ticket_ids` | `_is_valid_id()` | Internal by convention. Often called protected. Python allows outside access. |
| Two leading underscores | `__max_size` | `__has_room()` | Python changes the name inside the class. Often called private. This helps avoid name collisions in subclasses. |

In this example, other code calls the public method `add_ticket()`. The method uses two helpers to check the ID and the available space before it changes the queue.

In [35]:
class SupportQueue:
    def __init__(self, name: str):
        self.name = name
        self._ticket_ids: list[int] = []
        self.__max_size = 2

    def add_ticket(self, ticket_id: int) -> None:
        if self._is_valid_id(ticket_id) and self.__has_room():
            self._ticket_ids.append(ticket_id)

    def _is_valid_id(self, ticket_id: int) -> bool:
        return type(ticket_id) is int and ticket_id > 0

    def __has_room(self) -> bool:
        return len(self._ticket_ids) < self.__max_size

    def count(self) -> int:
        return len(self._ticket_ids)


queue = SupportQueue("login")
queue.add_ticket(101)
queue.add_ticket(102)
queue.add_ticket(103)
print(queue.name, queue.count())

login 2


**Expected** — `login 2`. The third ID passes validation, but the queue is full.

### Can we call an internal method from outside?

With one leading underscore, yes. Python lets us call the method. The underscore tells other developers it is a helper. They should normally call the public method, because that runs all the checks together.

- `queue._is_valid_id(103)` works from outside the class.
- `queue.__has_room()` raises `AttributeError`. Python stores that method under the changed name `_SupportQueue__has_room`. This is called **name mangling**.
- The changed name can still be accessed from outside, so name mangling is a naming mechanism. It provides no security boundary.
- Names like `__init__` have underscores at both ends. These are special method names Python uses for operations such as setting up an object.

In [36]:
print(queue._ticket_ids)
print(queue._is_valid_id(103))

try:
    print(queue.__has_room())
except AttributeError:
    print("Direct lookup raised AttributeError.")

# Demonstrate the changed name so students can see how mangling works.
print(queue._SupportQueue__has_room())
print(queue._SupportQueue__max_size)

[101, 102]
True
Direct lookup raised AttributeError.
False
2


**Expected** — `[101, 102]`, then `True`, then the message from the error handler, then `False` and `2`. The same rule applies to methods and attributes.

**Check your understanding** — If another part of the application needs to add a ticket, which method should it call?  
**Answer** — `add_ticket()`. It checks the ticket ID and the available space before adding anything.

### Singleton class

Suppose two parts of our support app need the same settings. If one changes the wait threshold from 30 to 45 minutes, the other should read 45 too. In this example, both parts use the same settings object. A class that gives back the same instance each time we call it uses the **singleton** pattern.

- `__new__()` chooses the instance to return when we call the class.
- The first time, `super().__new__(cls)` creates the instance. We set the initial threshold and save the object in `_instance`.
- On later calls, we return the saved instance.
- `first_settings is second_settings` checks whether both names point to the very same object.

In [37]:
class SupportSettings:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.threshold_minutes = 30

        return cls._instance


first_settings = SupportSettings()
second_settings = SupportSettings()

print(first_settings is second_settings)
print(second_settings.threshold_minutes)

first_settings.threshold_minutes = 45
print(second_settings.threshold_minutes)

True
30
45


**Expected** — `True`, then `30`, then `45`.

**Check your understanding** — Why does changing the first setting affect what we read from the second?  
**Answer** — Both names refer to the same object. We changed the one threshold stored on it.

**Common question · Why use `__new__()` here?**  
**Answer** — `__new__()` returns the object. `__init__()` sets up an object after Python has it. If we put `threshold_minutes = 30` in `__init__()`, another call to `SupportSettings()` would reset the shared value in this example.

**Common question · Is the object shared across notebooks?**  
**Answer** — It is shared by callers of this class in the same running Python process. A notebook with a separate kernel has its own instance.

**Agent connection** — Tools in one application can read shared app settings. Each tool call can keep its own ticket data.